# Lottery Ticket Hypothesis

The core technique we are trying to test here is that of the Lottery Ticket Hypothesis, which posits that within a randomly initialized neural network, there exists a smaller sub-network (the "winning ticket") that, if reinitialized with the same initial weights, can train as effectively as the full network. In essence, the hypothesis suggests that large neural networks contain sparse, trainable sub-networks that, when identified and trained from the same starting point, can achieve comparable performance to the original network with fewer resources and training time.

In [1]:
import os

target_folder = "CS6423_knowledge_distillation_project" 
path = os.path.join(os.getcwd(), target_folder)

if not os.getcwd().endswith(target_folder):
    os.chdir(path)

# should match the folder you cloned into
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/cc65/CS6423_knowledge_distillation_project


In [2]:
import sys
sys.path.append('.')

from modules import ImagenetLoader, datasetPrepper, modelTrainer, ModelEvaluator
import pandas as pd

pretrained_weights_path = "RadImageNet_weights/resnet50.pth"
dataframe_path = "data/labels.csv"
image_dir = "data/test_images"
model_name = "radimagenet50_vqa_baseline1"

dataframe = pd.read_csv(dataframe_path)
num_classes = dataframe["label"].nunique()

dataframe.head()

,filename,pathology,modality,location,label
0,test_0.png,flexor_pathology_,mri,ankle foot,ankle_foot_flexor_pathology_
1,test_1.png,flexor_pathology_,mri,ankle foot,ankle_foot_flexor_pathology_
2,test_2.png,flexor_pathology_,mri,ankle foot,ankle_foot_flexor_pathology_
3,test_3.png,flexor_pathology_,mri,ankle foot,ankle_foot_flexor_pathology_
4,test_4.png,flexor_pathology_,mri,ankle foot,ankle_foot_flexor_pathology_


In [12]:
import torch as T
import torch.nn.utils.prune as prune
from torchvision import models
import numpy as np

# Set up the Data
data = datasetPrepper(
    dataframe_path=dataframe_path,
    image_dir=image_dir,
).prepare(compute_class_weights=True)

print(f"Dataset prepared:")
print(f"  Train samples: {len(data.train_dataset)}")
print(f"  Val samples: {len(data.val_dataset)}")
print(f"  Classes: {len(data.class_names)}")

# Define the evaluator
evaluator = ModelEvaluator(data.val_loader, data.class_names)
eval_stats = {}

# Load and train up a baseline to compare
loader = ImagenetLoader()
loader.load_radimagenet_resnet50(pretrained_weights_path)
model = loader.model
trainer = modelTrainer(
    model=model,
    data_prep=data,
    device=None,
    learn_rate=0.001,
    num_epochs=10,
    model_name="baseline_comparison_model",
)
trainer.prepare_for_training(trainable_params=loader.get_trainable_params())
trainer.train_all()
eval_stats[model_name] = evaluator.evaluate_single(model)

# Define the pruning percentages
pruning_percentages = [10, 25, 50, 75, 90]
for p_percent in pruning_percentages:
    print(f"\nRunning LTH for {p_percent}% pruning")
    
    # Reset Weights at the start of every cycle
    loader.load_radimagenet_resnet50(pretrained_weights_path)
    model = loader.model
    
    # Train the parent model
    trainer = modelTrainer(
        model=model,
        data_prep=data,
        device=None,
        learn_rate=0.001,
        num_epochs=10,
        model_name=f"parent_${p_percent}",
    )
    trainer.prepare_for_training(trainable_params=loader.get_trainable_params())
    trainer.train_all()
    
    # Get the pruning mask from the parent model
    model.eval()
    masks = {}
    
    for name, module in model.named_modules():
      if isinstance(module, (T.nn.Linear, T.nn.Conv2d)):
        prune.l1_unstructured(module, name='weight', amount=p_percent / 100.0)
        masks[name] = (module.weight_mask).cpu()
        prune.remove(module, 'weight')

    # Reset Weights
    loader.load_radimagenet_resnet50(pretrained_weights_path)
    model = loader.model
    trainer = modelTrainer(
        model=model,
        data_prep=data,
        device=None,
        learn_rate=0.001,
        num_epochs=10,
        model_name=f"{model_name}_{p_percent}",
    )

    # Apply the mask by directly setting weights to zero where mask is zero
    with T.no_grad():
      for name, module in model.named_modules():
        if isinstance(module, (T.nn.Linear, T.nn.Conv2d)) and name in masks:
            module.weight.data.mul_(masks[name])
            prune.custom_from_mask(module, name='weight', mask=masks[name])
            prune.remove(module, 'weight')
    
    # Train that ticket
    print(f"Retraining winning ticket for {p_percent}% LTH pruning...")
    model.train()
    trainer.prepare_for_training(trainable_params=loader.get_trainable_params())
    trainer.train_all()

    # Evaluate that ticket
    model.eval()
    eval_stats[trainer.model_name] = evaluator.evaluate_single(model)
    
    print(f"\n======================\nDone for ${p_percent}\n======================\n")


Warming up Model...
Running inference...

Running LTH for 10% pruning


Validating: 100%|██████████| 57/57 [00:04<00:00, 14.08batch/s]



Epoch 1/10
Train Loss: 3.7602 | Train F1: 0.0506
Val Loss: 3.9147 | Val F1: 0.0386
Epoch Time: 34.54s



Validating: 100%|██████████| 57/57 [00:04<00:00, 14.09batch/s]



Epoch 2/10
Train Loss: 2.1203 | Train F1: 0.2023
Val Loss: 3.1084 | Val F1: 0.1255
Epoch Time: 34.38s



...


Epoch 8/10
Train Loss: 0.6238 | Train F1: 0.5975
Val Loss: 1.3965 | Val F1: 0.4771
Epoch Time: 34.44s



Validating: 100%|██████████| 57/57 [00:04<00:00, 14.04batch/s]



Epoch 9/10
Train Loss: 0.5200 | Train F1: 0.6446
Val Loss: 1.0555 | Val F1: 0.5144
Epoch Time: 34.41s



Validating: 100%|██████████| 57/57 [00:04<00:00, 13.96batch/s]



Epoch 10/10
Train Loss: 0.4078 | Train F1: 0.6919
Val Loss: 0.8645 | Val F1: 0.5749
Epoch Time: 34.42s


Warming up Model...
Running inference...
Again, again


In [13]:
for thing in eval_stats:
    print(thing,"\n", eval_stats[thing], "\n\n")

radimagenet50_vqa_baseline1 
 {'f1_micro': 0.0033333333333333335, 'f1_macro': 0.0005906342129882692, 'sensitivity': 0.00663544106167057, 'avg_latency_ms': 1.0977274916917343, 'model_size_mb': 90.83265686035156, 'device': 'cuda'} 


radimagenet50_vqa_baseline1_10 
 {'f1_micro': 0.5555555555555556, 'f1_macro': 0.706043040086498, 'sensitivity': 0.8656292486903312, 'avg_latency_ms': 1.0970879167628786, 'model_size_mb': 90.83265686035156, 'device': 'cuda'} 


radimagenet50_vqa_baseline1_25 
 {'f1_micro': 0.45055555555555554, 'f1_macro': 0.632730297162577, 'sensitivity': 0.803637126666453, 'avg_latency_ms': 1.096814094424998, 'model_size_mb': 90.83265686035156, 'device': 'cuda'} 


radimagenet50_vqa_baseline1_50 
 {'f1_micro': 0.4855555555555556, 'f1_macro': 0.6444553370110377, 'sensitivity': 0.8169019838361068, 'avg_latency_ms': 1.0971067200363096, 'model_size_mb': 90.83265686035156, 'device': 'cuda'} 


radimagenet50_vqa_baseline1_75 
 {'f1_micro': 0.34, 'f1_macro': 0.484464292363402, 'sen